# README
## Purpose
Train and evaluate an LDA baseline on combined timeline summaries.
## Inputs
- `Data/combined_timeline_data.csv`
## Outputs
- LDA topics, coherence diagnostics, and topic-word inspection results.
## Notes
This classical baseline supports fair comparison with transformer-based approaches.

Imports


In [1]:
import pandas as pd
import gensim
from gensim import corpora
from gensim.models import LdaModel, CoherenceModel
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import re

Load Data

In [2]:
df = pd.read_csv('Data/combined_timeline_data.csv')
df.head()

,Date,Source,Summary,Associated Link Title,Associated Link URL,Theme
0,"January 4, 2020",WHO Announces Pneumonia Cases of Unknown Cause,The World Health Organization (WHO) announces ...,WHO on Twitter,https://fraser.stlouisfed.org/docs/historical/...,Pandemic
1,"January 8, 2020",CDC Issues Health Advisory,The Centers for Disease Control and Prevention...,CDC Health Advisory,https://fraser.stlouisfed.org/archival-collect...,Pandemic
2,"January 9, 2020",CDC Notes Appearance of Novel Coronavirus Outb...,The CDC notes the appearance of a novel corona...,CDC Report,https://fraser.stlouisfed.org/archival-collect...,Pandemic
3,"January 17, 2020, 2:00 pm EST",CDC Announces Enhanced Screenings for Those Tr...,The CDC and the Department of Homeland Securit...,CDC Press Release,https://fraser.stlouisfed.org/archival-collect...,Pandemic
4,"January 21, 2020",Washington State Department of Health Announce...,The Washington State Department of Health anno...,Snohomish Health District Media Release,https://fraser.stlouisfed.org/title/state-mate...,Pandemic


Preprocess Data

In [3]:
docs = df["Summary"].tolist()
# Extract only the date part (remove everything after the year)
timestamps = pd.to_datetime(df["Date"].str.extract(r'([A-Za-z]+\s+\d{1,2},\s+\d{4})')[0]).dt.date

# 1. Preprocessing (Required for LDA)
nltk.download('stopwords')
nltk.download('wordnet')
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_lda(text):
    text = re.sub(r'\s+', ' ', text).lower() # Remove extra whitespace & lowercase
    tokens = [lemmatizer.lemmatize(w) for w in text.split() if w not in stop_words and len(w) > 3]
    return tokens

# Process your existing 'docs' list
processed_docs = [preprocess_lda(doc) for doc in docs]

# 2. Create Dictionary and Corpus
id2word = corpora.Dictionary(processed_docs)
# Filter out words that appear in less than 5 docs or more than 50% of docs
id2word.filter_extremes(no_below=5, no_above=0.5) 
corpus = [id2word.doc2bow(text) for text in processed_docs]

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\gianf\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\gianf\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


Train LDA Model

In [4]:
# 3. Train LDA Model
# Note: You MUST pre-define the number of topics (k)
num_topics = 12 
lda_model = LdaModel(corpus=corpus, id2word=id2word, num_topics=num_topics, 
                     random_state=42, passes=10, alpha='auto')

View Topics

In [5]:
# 4. View Topics
for idx, topic in lda_model.print_topics(-1):
    print(f"Topic: {idx} \nWords: {topic}")

Topic: 0 
Words: 0.038*"federal" + 0.032*"market" + 0.031*"liquidity" + 0.028*"facility" + 0.028*"reserve" + 0.024*"fund" + 0.022*"commercial" + 0.022*"money" + 0.019*"paper" + 0.018*"rule"
Topic: 1 
Words: 0.029*"temporary" + 0.029*"federal" + 0.029*"reserve" + 0.027*"u.s." + 0.025*"announces" + 0.023*"facility" + 0.021*"bank" + 0.018*"guarantee" + 0.018*"extension" + 0.016*"central"
Topic: 2 
Words: 0.030*"bank" + 0.029*"federal" + 0.021*"reserve" + 0.018*"loan" + 0.017*"commission" + 0.017*"release" + 0.016*"financial" + 0.015*"would" + 0.014*"economic" + 0.014*"banking"
Topic: 3 
Words: 0.056*"federal" + 0.045*"credit" + 0.037*"reserve" + 0.031*"rate" + 0.029*"loan" + 0.028*"board" + 0.021*"primary" + 0.019*"percent." + 0.017*"basis" + 0.017*"point"
Topic: 4 
Words: 0.042*"quarter" + 0.029*"institution" + 0.027*"bank" + 0.026*"asset" + 0.025*"capital" + 0.024*"billion" + 0.024*"announces" + 0.023*"fdic" + 0.019*"board" + 0.018*"policy"
Topic: 5 
Words: 0.091*"u.s." + 0.073*"purchas

Preparing validation

In [6]:
# LDA (Gensim)
lda_topics = [
    [word for word, _ in lda_model.show_topic(t, topn=10)] 
    for t in range(lda_model.num_topics)
]

In [7]:
from gensim.models.coherencemodel import CoherenceModel

def calculate_metrics(topics, corpus_docs, dictionary):
    # 1. Coherence (Cv)
    cm = CoherenceModel(topics=topics, texts=corpus_docs, dictionary=dictionary, coherence='c_v')
    coherence = cm.get_coherence()
    
    # 2. Topic Diversity
    all_words = [word for topic in topics for word in topic]
    unique_words = set(all_words)
    diversity = len(unique_words) / len(all_words)
    
    return coherence, diversity

# Example usage for LDA
# (Repeat for bertopic_topics and top2vec_topics)
cv_lda, div_lda = calculate_metrics(lda_topics, processed_docs, id2word)
print(f"LDA Coherence (C_v): {cv_lda:.4f}")
print(f"LDA Topic Diversity: {div_lda:.4f}")

LDA Coherence (C_v): 0.5557
LDA Topic Diversity: 0.6083


## Export outputs for cross-model comparison

In [8]:
from pathlib import Path
from datetime import datetime
import json
import numpy as np

pipeline_name = 'LDA'
required = ['df', 'docs', 'lda_model', 'corpus']
missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(f'Missing required variables: {missing}. Run the full notebook first.')

run_tag = datetime.now().strftime('%Y%m%d_%H%M%S')
base_dir = Path('Outputs') / 'Comparisons_03' / pipeline_name / run_tag
folders = {
    'tables': base_dir / 'tables',
    'row_level': base_dir / 'row_level',
    'models': base_dir / 'models',
    'meta': base_dir / 'meta'
}
for folder in folders.values():
    folder.mkdir(parents=True, exist_ok=True)


def save_df(df_obj, stem, folder):
    csv_path = folder / f'{stem}.csv'
    df_obj.to_csv(csv_path, index=False, encoding='utf-8-sig')
    return str(csv_path)

exported = {}
notes = []

# Topic-level tables
num_topics = int(lda_model.num_topics)
topic_terms_rows = []
for topic_id in range(num_topics):
    terms = lda_model.show_topic(topic_id, topn=10)
    for rank, (term, weight) in enumerate(terms, start=1):
        topic_terms_rows.append({
            'topic_id': int(topic_id),
            'term_rank': int(rank),
            'term': str(term),
            'weight': float(weight)
        })

topic_terms_df = pd.DataFrame(topic_terms_rows)
exported['topic_terms'] = save_df(topic_terms_df, 'topic_terms_long', folders['tables'])

# Row-level assignments from corpus
summary_col = 'Summary' if 'Summary' in df.columns else df.columns[0]
date_col = 'Date' if 'Date' in df.columns else None
n = min(len(corpus), len(docs), len(df))

row_df = df.iloc[:n].copy().reset_index(drop=True)
dominant_topics = []
for bow in corpus[:n]:
    topic_probs = lda_model.get_document_topics(bow, minimum_probability=0.0)
    dominant_topic = max(topic_probs, key=lambda x: x[1])[0] if topic_probs else -1
    dominant_topics.append(int(dominant_topic))

assignments_df = pd.DataFrame({
    'row_index': np.arange(n),
    'text': row_df[summary_col].astype(str),
    'topic': dominant_topics
})
if date_col is not None:
    assignments_df['date'] = row_df[date_col].astype(str)

exported['row_topic_assignments'] = save_df(assignments_df, 'row_topic_assignments', folders['row_level'])

# Topic prevalence
topic_prevalence_df = (
    assignments_df.groupby('topic', as_index=False)
    .size()
    .rename(columns={'size': 'documents'})
    .sort_values('documents', ascending=False)
)
exported['topic_prevalence'] = save_df(topic_prevalence_df, 'topic_prevalence', folders['tables'])

# Metric summary
metric_rows = []
if 'cv_lda' in globals():
    metric_rows.append({'metric': 'coherence_c_v', 'value': float(cv_lda)})
if 'div_lda' in globals():
    metric_rows.append({'metric': 'topic_diversity', 'value': float(div_lda)})
if metric_rows:
    metrics_df = pd.DataFrame(metric_rows)
    exported['metrics'] = save_df(metrics_df, 'metrics_summary', folders['tables'])

# Model save
model_path = folders['models'] / 'lda_model.gensim'
try:
    lda_model.save(str(model_path))
    exported['model_file'] = str(model_path)
except Exception as exc:
    notes.append(f'Model save skipped: {exc}')

summary = {
    'pipeline': pipeline_name,
    'run_tag': run_tag,
    'export_root': str(base_dir),
    'saved_items': sorted(exported.keys()),
    'paths': exported,
    'notes': notes
}
summary_path = folders['meta'] / 'export_summary.json'
with open(summary_path, 'w', encoding='utf-8') as handle:
    json.dump(summary, handle, indent=2, ensure_ascii=False)

print(f'Export complete: {summary_path}')
print('Saved items:', ', '.join(sorted(exported.keys())) if exported else 'none')
if notes:
    print('Notes:')
    for note in notes:
        print('-', note)

Export complete: Outputs\Comparisons_03\LDA\20260416_213414\meta\export_summary.json
Saved items: metrics, model_file, row_topic_assignments, topic_prevalence, topic_terms
